# CrewAI Financial Researcher: 기업 분석 멀티 에이전트 프로젝트

이번 노트북에서는 CrewAI로 만든 **Financial Researcher** 프로젝트를 분석합니다. 웹 검색 도구(SerperDevTool)를 활용하여 기업 정보를 조사하고, 분석 리포트를 자동 생성하는 2-에이전트 시스템입니다.

## 개요

| 주제 | 내용 |
|------|------|
| 프로젝트 구조 | CrewAI 표준 프로젝트 레이아웃 |
| agents.yaml | researcher, analyst 에이전트 설정 |
| tasks.yaml | research_task, analysis_task 설정 + 명시적 context |
| crew.py | 데코레이터 패턴 + SerperDevTool 연결 |
| 직접 실행 | 노트북에서 Crew를 구성하여 실행 |

## 학습 목표

1. 실제 도구(SerperDevTool)를 Agent에 연결하는 방법 이해하기
2. `context` 파라미터로 태스크 간 명시적 정보 전달 패턴 확인하기
3. Debate 프로젝트와 비교하여 CrewAI 프로젝트의 공통 패턴과 차이점 파악하기
4. Financial Researcher Crew를 직접 실행하고 리포트 결과 확인하기

---

## Debate vs Financial Researcher 비교

| | Debate (03-2) | Financial Researcher (이번) |
|---|---|---|
| **에이전트 수** | 2 (debater, judge) | 2 (researcher, analyst) |
| **태스크 수** | 3 (propose, oppose, decide) | 2 (research, analysis) |
| **도구 사용** | 없음 | SerperDevTool (웹 검색) |
| **context 방식** | sequential 자동 전달 | 명시적 `context: [research_task]` |
| **출력 파일** | 태스크별 3개 파일 | 최종 리포트 1개 (output/report.md) |
| **변수** | `{motion}` | `{company}` |

---

## 1. 프로젝트 구조

```
financial_researcher/
├── pyproject.toml                  # crewai[tools]==1.6.1 의존성
├── .env                            # MODEL=gpt-4o-mini
├── knowledge/
│   └── user_preference.txt         # Knowledge 소스 (RAG용)
├── output/
│   └── report.md                   # 최종 분석 리포트
└── src/financial_researcher/
    ├── config/
    │   ├── agents.yaml             # 에이전트 정의
    │   └── tasks.yaml              # 태스크 정의
    ├── tools/
    │   └── custom_tool.py          # 커스텀 도구 템플릿
    ├── crew.py                     # Crew 클래스
    └── main.py                     # 실행 진입점
```

### 프로젝트의 아이디어

주어진 **기업(company)**에 대해:
1. **researcher** 에이전트가 웹 검색으로 기업 정보를 수집 (research_task)
2. **analyst** 에이전트가 수집된 정보를 분석하여 리포트 작성 (analysis_task)

---

## 2. agents.yaml 분석

```yaml
researcher:
  role: >
    Senior Financial Researcher for {company}
  goal: >
    Research the company, news and potential for {company}
  backstory: >
    You're a seasoned financial researcher with a talent for finding
    the most relevant information about {company}.
    Known for your ability to find the most relevant
    information and present it in a clear and concise manner.
  llm: openai/gpt-4o-mini

analyst:
  role: >
    Market Analyst and Report writer focused on {company}
  goal: >
    Analyze company {company} and create a comprehensive,
    well-structured report that presents insights in a clear
    and engaging way
  backstory: >
    You're a meticulous, skilled analyst with a background
    in financial analysis and company research. You have a
    talent for identifying patterns and extracting meaningful
    insights from research data, then communicating those
    insights through well crafted reports.
  llm: openai/gpt-4o-mini
```

**Debate와의 차이점**: Debate에서는 같은 debater가 찬성/반대를 모두 수행했지만, 여기서는 **역할이 명확히 분리**되어 있습니다 — researcher는 조사만, analyst는 분석/작성만 담당합니다.

---

## 3. tasks.yaml 분석

```yaml
research_task:
  description: >
    Conduct thorough research on company {company}. Focus on:
    1. Current company status and health
    2. Historical company performance
    3. Major challenges and opportunities
    4. Recent news and events
    5. Future outlook and potential developments

    Make sure to organize your findings in a structured format
    with clear sections.
  expected_output: >
    A comprehensive research document with well-organized sections
    covering all the requested aspects of {company}. Include specific
    facts, figures, and examples where relevant.
  agent: researcher

analysis_task:
  description: >
    Analyze the research findings and create a comprehensive report
    on {company}. Your report should:
    1. Begin with an executive summary
    2. Include all key information from the research
    3. Provide insightful analysis of trends and patterns
    4. Offer a market outlook for company
    5. Be formatted in a professional, easy-to-read style
  expected_output: >
    A polished, professional report on {company} that presents
    the research findings with added analysis and insights.
  agent: analyst
  context:
    - research_task          # ← 명시적 context 지정!
  output_file: output/report.md
```

### 주목할 점: 명시적 `context` 사용

```
┌────────────────────────────────────────────────────────────┐
│           context를 사용한 태스크 간 정보 전달              │
├────────────────────────────────────────────────────────────┤
│                                                            │
│  research_task (researcher)                                │
│    │  SerperDevTool로 웹 검색                              │
│    │  기업 현황, 실적, 뉴스 등 수집                        │
│    │                                                       │
│    │  context: [research_task]                              │
│    ▼                                                       │
│  analysis_task (analyst)                                   │
│    │  research_task의 결과를 참고하여                       │
│    │  종합 리포트 작성                                     │
│    ▼                                                       │
│  output/report.md                                          │
│                                                            │
└────────────────────────────────────────────────────────────┘
```

이 프로젝트는 sequential 프로세스이므로 context를 명시하지 않아도 자동 전달됩니다. 그러나 YAML에서 `context: [research_task]`를 **명시적으로 선언**하여 의도를 더 명확하게 표현하고 있습니다. 태스크가 많아질수록 이런 명시적 선언이 가독성과 유지보수에 유리합니다.

---

## 4. crew.py 분석

```python
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool

@CrewBase
class ResearchCrew():
    """Research crew for comprehensive topic analysis and reporting"""

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'],
            verbose=True,
            tools=[SerperDevTool()]    # ← 웹 검색 도구 연결!
        )

    @agent
    def analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['analyst'],
            verbose=True
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task']
        )

    @task
    def analysis_task(self) -> Task:
        return Task(
            config=self.tasks_config['analysis_task'],
            output_file='output/report.md'
        )

    @crew
    def crew(self) -> Crew:
        """Creates the research crew"""
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
        )
```

### Debate와 다른 핵심 포인트

| 차이점 | Debate | Financial Researcher |
|--------|--------|---------------------|
| **도구 사용** | 없음 | `tools=[SerperDevTool()]` |
| **YAML에 agents/tasks_config 경로** | 명시적 선언 | `@CrewBase` 기본값 사용 (생략) |
| **output_file** | YAML에서 설정 | crew.py 코드에서 설정 |

`agents_config`와 `tasks_config` 경로를 생략하면 `@CrewBase`가 기본 경로(`config/agents.yaml`, `config/tasks.yaml`)를 자동으로 사용합니다.

---

## 5. SerperDevTool — 웹 검색 도구

이 프로젝트의 핵심 차별점은 **researcher 에이전트가 실제로 웹 검색**을 수행한다는 것입니다.

### SerperDevTool이란?

[Serper](https://serper.dev/)는 Google 검색 결과를 API로 제공하는 서비스입니다. `crewai_tools` 패키지에 포함된 `SerperDevTool`을 통해 에이전트가 실시간 웹 검색을 수행할 수 있습니다.

```
┌─────────────────────────────────────────────────────────────────────┐
│              SerperDevTool 동작 흐름                                │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  researcher Agent                                                   │
│    │                                                                │
│    │ "Apple company 2024 financial performance" 검색 요청           │
│    ▼                                                                │
│  SerperDevTool                                                      │
│    │                                                                │
│    │ Serper API → Google 검색 결과 반환                             │
│    ▼                                                                │
│  researcher Agent                                                   │
│    │                                                                │
│    │ 검색 결과를 종합하여 리서치 문서 작성                          │
│    │ (필요하면 추가 검색 반복)                                       │
│    ▼                                                                │
│  research_task 결과물                                               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 사용을 위한 준비

SerperDevTool을 사용하려면 [serper.dev](https://serper.dev/)에서 API 키를 발급받고 `.env`에 추가해야 합니다:

```
SERPER_API_KEY=your_key_here
```

> SerperDevTool 없이도 실행 가능합니다. 아래 코드 셀에서 도구 없이 실행하는 방법도 함께 제공합니다.

---

## 6. main.py 분석

```python
import os
from financial_researcher.crew import ResearchCrew

os.makedirs('output', exist_ok=True)

def run():
    inputs = {
        'company': 'Apple'
    }
    result = ResearchCrew().crew().kickoff(inputs=inputs)

    print("\n\n=== FINAL REPORT ===\n\n")
    print(result.raw)
    print("\n\nReport has been saved to output/report.md")
```

- `inputs`의 `company` 값이 YAML의 모든 `{company}` 자리에 삽입됩니다
- CLI에서는 `crewai run` 또는 `uv run financial_researcher`로 실행합니다

### 전체 실행 흐름

```
┌─────────────────────────────────────────────────────────────────┐
│           Financial Researcher 실행 흐름                        │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  inputs = {'company': 'Apple'}                                  │
│       │                                                         │
│       ▼                                                         │
│  ┌───────────────────┐                                          │
│  │  1. research_task  │  Agent: researcher                      │
│  │                    │  Tool: SerperDevTool (웹 검색)           │
│  │                    │  기업 현황, 실적, 뉴스, 전망 조사        │
│  └────────┬──────────┘                                          │
│           │ context 전달                                         │
│           ▼                                                      │
│  ┌───────────────────┐                                           │
│  │  2. analysis_task  │  Agent: analyst                          │
│  │                    │  executive summary + 분석 + 전망         │
│  │                    │  Output → output/report.md               │
│  └───────────────────┘                                           │
│           │                                                      │
│           ▼                                                      │
│    CrewOutput (최종 리포트)                                       │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

---

## 7. Financial Researcher Crew 직접 실행

노트북에서 Python 코드로 동일한 Crew를 구성하여 실행합니다.

> **참고**: `SERPER_API_KEY`가 없으면 SerperDevTool 없이 실행됩니다. 도구 없이도 LLM의 내부 지식만으로 리서치를 수행합니다.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")

from dotenv import load_dotenv
import os

load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')
serper_key = os.getenv('SERPER_API_KEY')

print(f"OPENAI_API_KEY: {'found' if openai_key else 'NOT FOUND — .env 파일에 설정하세요'}")
print(f"SERPER_API_KEY: {'found' if serper_key else 'NOT FOUND — 웹 검색 없이 실행됩니다'}")

OPENAI_API_KEY: found
SERPER_API_KEY: found


In [2]:
from crewai import Agent, Task, Crew, Process

# SerperDevTool이 사용 가능한지 확인
researcher_tools = []
if os.getenv('SERPER_API_KEY'):
    from crewai_tools import SerperDevTool
    researcher_tools = [SerperDevTool()]
    print("SerperDevTool 활성화 — 웹 검색을 사용합니다.")
else:
    print("SerperDevTool 비활성화 — LLM 내부 지식만 사용합니다.")

# 분석할 기업
company = "Apple"

# Agent 정의 — agents.yaml과 동일
researcher = Agent(
    role=f"Senior Financial Researcher for {company}",
    goal=f"Research the company, news and potential for {company}",
    backstory=(
        f"You're a seasoned financial researcher with a talent for finding "
        f"the most relevant information about {company}. Known for your ability to find "
        f"the most relevant information and present it in a clear and concise manner."
    ),
    verbose=True,
    llm="openai/gpt-4o-mini",
    tools=researcher_tools,
)

analyst = Agent(
    role=f"Market Analyst and Report writer focused on {company}",
    goal=(
        f"Analyze company {company} and create a comprehensive, well-structured report "
        f"that presents insights in a clear and engaging way"
    ),
    backstory=(
        "You're a meticulous, skilled analyst with a background in financial analysis "
        "and company research. You have a talent for identifying patterns and extracting "
        "meaningful insights from research data, then communicating those insights "
        "through well crafted reports."
    ),
    verbose=True,
    llm="openai/gpt-4o-mini",
)

print(f"에이전트 생성 완료: {researcher.role}, {analyst.role}")

ModuleNotFoundError: No module named 'crewai_tools'

In [ ]:
# Task 정의 — tasks.yaml과 동일

research_task = Task(
    description=(
        f"Conduct thorough research on company {company}. Focus on:\n"
        f"1. Current company status and health\n"
        f"2. Historical company performance\n"
        f"3. Major challenges and opportunities\n"
        f"4. Recent news and events\n"
        f"5. Future outlook and potential developments\n\n"
        f"Make sure to organize your findings in a structured format with clear sections."
    ),
    expected_output=(
        f"A comprehensive research document with well-organized sections covering "
        f"all the requested aspects of {company}. Include specific facts, figures, "
        f"and examples where relevant."
    ),
    agent=researcher,
)

analysis_task = Task(
    description=(
        f"Analyze the research findings and create a comprehensive report on {company}.\n"
        f"Your report should:\n"
        f"1. Begin with an executive summary\n"
        f"2. Include all key information from the research\n"
        f"3. Provide insightful analysis of trends and patterns\n"
        f"4. Offer a market outlook for company, noting that this should not be used for trading decisions\n"
        f"5. Be formatted in a professional, easy-to-read style with clear headings"
    ),
    expected_output=(
        f"A polished, professional report on {company} that presents the research "
        f"findings with added analysis and insights. The report should be well-structured "
        f"with an executive summary, main sections, and conclusion."
    ),
    agent=analyst,
    context=[research_task],       # ← 명시적 context 지정
)

print("태스크 생성 완료: research_task, analysis_task")
print(f"analysis_task의 context: research_task (연구 결과를 참고)")

In [ ]:
# Crew 구성 및 실행

research_crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
)

result = research_crew.kickoff()

print("\n" + "="*60)
print("FINAL REPORT")
print("="*60)
print(result.raw)

In [ ]:
# 리포트를 rich로 보기 좋게 출력

from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# 리서치 결과
console.print(Panel(
    Markdown(result.tasks_output[0].raw),
    title=f"Research: {company}",
    border_style="cyan",
))
console.print()

# 최종 분석 리포트
console.print(Panel(
    Markdown(result.tasks_output[1].raw),
    title=f"Analysis Report: {company}",
    border_style="green",
))

---

## 정리

### Financial Researcher 프로젝트에서 배운 것

```
┌─────────────────────────────────────────────────────────────────────┐
│           Financial Researcher 핵심 요약                           │
├──────────────────────────────┬──────────────────────────────────────┤
│     개념                     │     이 프로젝트에서의 적용           │
├──────────────────────────────┼──────────────────────────────────────┤
│  Tool 연결                   │  SerperDevTool → researcher Agent   │
│  명시적 context              │  analysis_task → [research_task]    │
│  output_file                 │  analysis_task → output/report.md   │
│  @CrewBase 기본값            │  config 경로 생략 (자동 탐색)       │
│  변수 치환                   │  {company} → inputs['company']      │
└──────────────────────────────┴──────────────────────────────────────┘
```

### 새로 배운 패턴

- **도구 연결**: `tools=[SerperDevTool()]`로 에이전트에게 실제 웹 검색 능력 부여
- **명시적 context**: YAML에서 `context: [research_task]`로 태스크 간 의존 관계를 명확하게 선언
- **역할 분리**: researcher(조사) + analyst(분석) 각각 전문 분야에 집중
- **@CrewBase 기본값**: `agents_config`/`tasks_config` 경로를 생략하면 기본 경로 자동 사용

### CLI로 실행하기

```bash
cd agent_engineering/03_crew/financial_researcher
uv sync
crewai run
# 또는: uv run financial_researcher
```